# Module 4: Recommendation Engine Demonstration
### Infosys Springboard Internship 7.0 — Milestone 2 Deliverable
**Author:** S Koushick  
**Target Completion:** 6 November 2026

---
### Notebook Objectives
1. **Load Production Recommendation Artifacts:** Load fitted TF-IDF vectorizers (44,613 features), dense SentenceTransformer embeddings (384 dimensions), and 2550 TED talks.
2. **Demonstrate Content-Based TF-IDF Recommender:** Query talks by seed talk ID with metadata and tag filtering.
3. **Demonstrate Pretrained Semantic Recommender:** Discover conceptual and thematic matches beyond keyword overlap.
4. **Demonstrate Multi-Signal Hybrid Recommender:** Blend lexical, semantic, and topic signals with configurable weighting.
5. **Session-Based Personalisation:** Recommend based on multiple liked talks, negative exclusions (disliked talks), and user-selected interests.
6. **Maximal Marginal Relevance (MMR) Diversification:** Balance recommendation accuracy with content diversity.
7. **Explainability Engine:** Provide clear, auditable rationales for every recommended talk.
8. **Cold-Start & Edge Cases:** Demonstrate graceful fallback for users with no history and talks without transcripts.
9. **Collaborative Filtering Feasibility Audit:** Formally document dataset limitations and future interface specifications.


In [1]:
import sys
from pathlib import Path

# Automatically locate and register project root
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import json

from ted_recommender.config import RecommenderConfig
from ted_recommender.data import load_dataset, validate_dataset
from ted_recommender.features import FeaturePipeline
from ted_recommender.recommender import (
    ContentBasedRecommender,
    SemanticRecommender,
    HybridRecommender,
)
from ted_recommender.collaborative import assess_collaborative_filtering_feasibility

print('Recommender libraries imported successfully!')
print('Project root:', PROJECT_ROOT)


Recommender libraries imported successfully!
Project root: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation


## 1. Load Dataset and Production Artifacts


In [1]:
# Load feature dataset
df = load_dataset(stage='features')
print('Dataset shape:', df.shape, '(2550 talks verified)')

# Load configuration and feature pipeline artifacts
config = RecommenderConfig()
pipeline = FeaturePipeline(tfidf_config=config.tfidf_config, field_weights=config.field_weights)
pipeline.load_artifacts()

print('TF-IDF Matrix shape:', pipeline.combined_tfidf_matrix.shape)
print('Semantic Embeddings shape:', pipeline.embedding_matrix.shape)

# Initialize recommenders
tfidf_rec = ContentBasedRecommender(df, pipeline, config)
semantic_rec = SemanticRecommender(df, pipeline, config)
hybrid_rec = HybridRecommender(df, pipeline, config)
print('All recommenders initialized successfully!')


Dataset shape: (2550, 60) (2550 talks verified)
TF-IDF Matrix shape: (2550, 44613)
Semantic Embeddings shape: (2550, 384)
All recommenders initialized successfully!


## 2. Content-Based TF-IDF Recommender
Recommending talks similar to **Sir Ken Robinson: 'Do schools kill creativity?' (Talk ID: 1)**


In [1]:
results_tfidf = tfidf_rec.recommend_for_talk(talk_id=1, top_k=5)

for i, res in enumerate(results_tfidf, start=1):
    print(f'{i}. {res.title} (Speaker: {res.main_speaker})')
    print(f'   Score: {res.score:.4f} | Views: {res.views:,}')
    print(f'   Rationale: {res.explanation.rationale}')
    print(f'   URL: {res.url}\n')


1. Bring on the learning revolution! (Speaker: Ken Robinson)
   Score: 0.2333 | Views: 7,266,316
   Rationale: Shared themes: children, creativity, education; Same speaker (Ken Robinson); Lexical content similarity: 0.233
   URL: https://www.ted.com/talks/sir_ken_robinson_bring_on_the_revolution

2. How to escape education's death valley (Speaker: Ken Robinson)
   Score: 0.1857 | Views: 6,657,858
   Rationale: Shared themes: culture, education, teaching; Same speaker (Ken Robinson); Lexical content similarity: 0.186
   URL: https://www.ted.com/talks/ken_robinson_how_to_escape_education_s_death_valley

3. Authentic creativity vs. karaoke culture (Speaker: Malcolm McLaren)
   Score: 0.1810 | Views: 251,313
   Rationale: Shared themes: creativity, culture; Lexical content similarity: 0.181
   URL: https://www.ted.com/talks/malcolm_mclaren_authentic_creativity_vs_karaoke_culture

4. Kids need structure (Speaker: Colin Powell)
   Score: 0.1648 | Views: 1,485,801
   Rationale: Shared themes:

## 3. Semantic Embedding Recommender
Capturing conceptual nuance using **all-MiniLM-L6-v2** sentence transformers.


In [1]:
results_sem = semantic_rec.recommend_for_talk(talk_id=1, top_k=5)

for i, res in enumerate(results_sem, start=1):
    print(f'{i}. {res.title} (Speaker: {res.main_speaker})')
    print(f'   Score: {res.score:.4f} | Views: {res.views:,}')
    print(f'   Rationale: {res.explanation.rationale}\n')


1. How art, technology and design inform creative leaders (Speaker: John Maeda)
   Score: 0.6462 | Views: 1,045,703
   Rationale: Semantic embedding similarity: 0.646; Shared themes: creativity

2. A short intro to the Studio School (Speaker: Geoff Mulgan)
   Score: 0.6220 | Views: 667,985
   Rationale: Semantic embedding similarity: 0.622; Shared themes: creativity, culture, education

3. Do the green thing (Speaker: Andy Hobsbawm)
   Score: 0.6152 | Views: 429,967
   Rationale: Semantic embedding similarity: 0.615

4. Teach arts and sciences together (Speaker: Mae Jemison)
   Score: 0.5897 | Views: 744,257
   Rationale: Semantic embedding similarity: 0.590; Shared themes: dance, education

5. Where does creativity hide? (Speaker: Amy Tan)
   Score: 0.5879 | Views: 2,477,746
   Rationale: Semantic embedding similarity: 0.588; Shared themes: culture



## 4. Personalised Hybrid Recommender with Multi-Liked & Disliked Talks
Scenario: User likes talks on **creativity & climate**, dislikes talk on **simplicity**, and expresses topical interest in **education & design**.


In [1]:
results_hybrid = hybrid_rec.recommend(
    liked_talk_ids=[1, 2],         # Ken Robinson (Creativity) + Al Gore (Climate)
    disliked_talk_ids=[3],         # Dislikes David Pogue (Simplicity)
    user_interests=['education', 'design'],
    top_k=5,
    apply_mmr=True,
    tfidf_weight=0.4,
    semantic_weight=0.6,
)

for i, res in enumerate(results_hybrid, start=1):
    print(f'{i}. {res.title} (Speaker: {res.main_speaker})')
    print(f'   Hybrid Score: {res.score:.4f}')
    print(f'   Rationale: {res.explanation.rationale}')
    print(f'   Tags: {res.tags[:5]}\n')


1. How I teach kids to love science (Speaker: Cesar Harada)
   Hybrid Score: 0.5700
   Rationale: Matching topics: design, education, environment, global issues; Hybrid Score: 0.570 (TF-IDF: 0.06, Semantic: 0.58)
   Tags: ['ted fellows', 'collaboration', 'design', 'education', 'environment']

2. Tales of creativity and play (Speaker: Tim Brown)
   Hybrid Score: 0.5523
   Rationale: Matching topics: creativity, design, education; Hybrid Score: 0.552 (TF-IDF: 0.09, Semantic: 0.53)
   Tags: ['art', 'creativity', 'design', 'education', 'play']

3. My wish: Once Upon a School (Speaker: Dave Eggers)
   Hybrid Score: 0.5366
   Rationale: Matching topics: children, culture, design, education; Hybrid Score: 0.537 (TF-IDF: 0.08, Semantic: 0.51)
   Tags: ['ted prize', 'activism', 'children', 'collaboration', 'culture']

4. A short intro to the Studio School (Speaker: Geoff Mulgan)
   Hybrid Score: 0.5385
   Rationale: Matching topics: creativity, culture, design, education; Hybrid Score: 0.538 (T

## 5. Maximal Marginal Relevance (MMR) Diversification
Comparing standard top-K ranking vs. MMR diversified ranking.


In [1]:
# Standard top-K ranking (no MMR)
std_results = hybrid_rec.recommend(seed_talk_id=1, top_k=5, apply_mmr=False)
# MMR diversified ranking (lambda = 0.7)
mmr_results = hybrid_rec.recommend(seed_talk_id=1, top_k=5, apply_mmr=True)

print('--- Standard Top-5 Ranking ---')
for r in std_results:
    print(f'- {r.title} ({r.score:.4f})')

print('\n--- MMR Diversified Top-5 Ranking ---')
for r in mmr_results:
    print(f'- {r.title} ({r.score:.4f})')


--- Standard Top-5 Ranking ---
- A short intro to the Studio School (0.3696)
- Authentic creativity vs. karaoke culture (0.3682)
- Teach arts and sciences together (0.3631)
- How to escape education's death valley (0.3563)
- Our failing schools. Enough is enough! (0.3535)

--- MMR Diversified Top-5 Ranking ---
- A short intro to the Studio School (0.3696)
- Where does creativity hide? (0.3358)
- How to escape education's death valley (0.3563)
- Bring on the learning revolution! (0.3497)
- Teach arts and sciences together (0.3631)


## 6. Cold-Start and Edge-Case Handling


In [1]:
# Cold-start fallback: User has zero history or input
cold_start_recs = hybrid_rec.recommend(top_k=3)
print('Cold Start Top-3:')
for r in cold_start_recs:
    print(f'- {r.title} | Views: {r.views:,} | Rationale: {r.explanation.rationale}')

# Safe edge-case handling for talks without transcripts
missing_transcript_talk = df[~df['has_transcript']].iloc[0]
print(f"\nTesting talk with missing transcript: Talk ID {missing_transcript_talk['talk_id']} ('{missing_transcript_talk['title']}')")
res_no_t = hybrid_rec.recommend(seed_talk_id=int(missing_transcript_talk['talk_id']), top_k=3)
print(f'Successfully generated {len(res_no_t)} recommendations using metadata/description fallback!')
for r in res_no_t:
    print(f'- {r.title} (Score: {r.score:.4f})')


Cold Start Top-3:
- Do schools kill creativity? | Views: 47,227,110 | Rationale: Cold-start fallback: Top trending TED talk by view count.
- Your body language may shape who you are | Views: 43,155,405 | Rationale: Cold-start fallback: Top trending TED talk by view count.
- How great leaders inspire action | Views: 34,309,432 | Rationale: Cold-start fallback: Top trending TED talk by view count.

Testing talk with missing transcript: Talk ID 59 ('A dance of "Symbiosis"')
Successfully generated 3 recommendations using metadata/description fallback!
- Dance to change the world (Score: 0.3158)
- Dance vs. powerpoint, a modest proposal (Score: 0.2861)
- A dance to honor Mother Earth (Score: 0.3140)


## 7. Collaborative Filtering Feasibility Assessment


In [1]:
cf_audit = assess_collaborative_filtering_feasibility(df)
print('Verdict:', cf_audit.verdict)
print('Rationale:', cf_audit.rationale)
print('\nFuture Interaction Schema Definition:')
print(json.dumps(cf_audit.future_schema_definition, indent=2))


Verdict: UNSUPPORTED_DATA_ABSENT
Rationale: The current TED Talk dataset provides only aggregate talk-level metrics (view counts, comment counts, and aggregate rating emotion tallies). No individual user identifiers or user-item interaction histories exist. In accordance with research integrity standards, no synthetic users or interactions are fabricated. Collaborative filtering cannot be trained until user-level logs are collected.

Future Interaction Schema Definition:
{
  "user_id": "string (unique anonymous user identifier)",
  "talk_id": "integer (foreign key matching talk_id in TED talks master catalog)",
  "interaction_type": "string (view, bookmark, like, share, completed_watch)",
  "watch_duration_seconds": "integer (dwell time on the talk video)",
  "rating": "optional float [1.0 - 5.0] or explicit binary thumbs_up/down",
  "timestamp": "ISO-8601 UTC timestamp"
}
